# 🧗 Cliff Walking Problem using SARSA Algorithm

This notebook demonstrates the implementation of the **SARSA (State-Action-Reward-State-Action)** Reinforcement Learning algorithm to solve the classic **Cliff Walking** environment from OpenAI / Gymnasium (`CliffWalking-v1`).

---

## 📖 1. Environment & Problem Description

The **Cliff Walking** environment is a 4x12 gridworld:
- **Grid Size**: 48 states (rows 0 to 3, columns 0 to 11).
- **Start State**: State 36 (bottom-left corner `[3, 0]`).
- **Goal State**: State 47 (bottom-right corner `[3, 11]`).
- **The Cliff**: States 37 to 46 (bottom row between Start and Goal).
- **Actions**: 4 discrete actions:
  - `0`: Up (↑)
  - `1`: Right (→)
  - `2`: Down (↓)
  - `3`: Left (←)

### Rewards:
- **Normal Transition**: -1 per step.
- **Stepping off the Cliff**: -100 penalty and immediate reset back to Start State (State 36).
- **Goal**: Terminal state.

## 🧠 2. SARSA Algorithm Overview

**SARSA** is an **On-Policy Temporal Difference (TD) control** algorithm.

### Update Rule:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \right]$$

Where:
- $S_t, A_t$: Current state and action.
- $R_{t+1}$: Reward received after taking action $A_t$.
- $S_{t+1}, A_{t+1}$: Next state and next action (selected using the current $\epsilon$-greedy policy).
- $\alpha$: Learning rate (0.5).
- $\gamma$: Discount factor (0.99).

> **Why SARSA for Cliff Walking?**  
> Because SARSA is **on-policy**, it accounts for the exploratory moves of the $\epsilon$-greedy strategy. As a result, SARSA learns a **safer route** along the top of the cliff rather than taking the risky shortest path directly along the edge.

In [ ]:
# Install dependencies (uncomment if running in fresh env)
# !pip install gymnasium numpy matplotlib pandas

In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd

## ⚙️ 3. Initialize Environment & Hyperparameters

In [ ]:
env = gym.make("CliffWalking-v1")

n_states = env.observation_space.n   # 48 states
n_actions = env.action_space.n      # 4 actions

print(f"Number of states: {n_states}")
print(f"Number of actions: {n_actions}")

# Hyperparameters
alpha = 0.5        # Learning rate
gamma = 0.99       # Discount factor
epsilon = 0.1      # Exploration rate
episodes = 500     # Training episodes

## 🛠️ 4. Define Epsilon-Greedy Policy & Q-Table

In [ ]:
# Initialize Q-Table with zeros
Q = np.zeros((n_states, n_actions))

def epsilon_greedy(state, Q_table, eps=epsilon):
    """
    Select an action using the Epsilon-Greedy strategy.
    """
    if random.random() < eps:
        return env.action_space.sample()
    else:
        max_val = np.max(Q_table[state])
        best_actions = np.where(Q_table[state] == max_val)[0]
        return int(random.choice(best_actions))

## 🚀 5. Training Loop (SARSA)

In [ ]:
rewards_history = []
lengths_history = []

for episode in range(episodes):
    state, _ = env.reset()
    action = epsilon_greedy(state, Q)
    
    total_reward = 0
    episode_length = 0
    done = False
    
    while not done:
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        next_action = epsilon_greedy(next_state, Q)
        
        # SARSA TD Update Rule
        target = reward
        if not done:
            target += gamma * Q[next_state, next_action]
            
        Q[state, action] += alpha * (target - Q[state, action])
        
        state = next_state
        action = next_action
        
        total_reward += reward
        episode_length += 1
        
    rewards_history.append(total_reward)
    lengths_history.append(episode_length)
    
    if (episode + 1) % 50 == 0 or episode == 0:
        print(f"Episode {episode+1:3d}/{episodes} | Total Reward: {total_reward:4d} | Episode Length: {episode_length:3d}")

env.close()

## 📊 6. Visualize Training Performance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

window = 20
rewards_ma = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
lengths_ma = np.convolve(lengths_history, np.ones(window)/window, mode='valid')

ax1.plot(rewards_history, alpha=0.3, color='steelblue', label='Raw Reward')
ax1.plot(range(window-1, len(rewards_history)), rewards_ma, color='darkblue', linewidth=2, label=f'{window}-Episode MA')
ax1.set_title("Total Reward per Episode")
ax1.set_xlabel("Episode")
ax1.set_ylabel("Reward")
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.plot(lengths_history, alpha=0.3, color='coral', label='Raw Steps')
ax2.plot(range(window-1, len(lengths_history)), lengths_ma, color='firebrick', linewidth=2, label=f'{window}-Episode MA')
ax2.set_title("Steps to Goal per Episode")
ax2.set_xlabel("Episode")
ax2.set_ylabel("Steps")
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

## 🗺️ 7. Visualize Learned Policy Grid Map

In [ ]:
shape = (4, 12)
fig, ax = plt.subplots(figsize=(12, 4))
action_symbols = {0: '↑', 1: '→', 2: '↓', 3: '←'}

ax.set_xlim(-0.5, shape[1] - 0.5)
ax.set_ylim(-0.5, shape[0] - 0.5)
ax.invert_yaxis()
ax.set_xticks(np.arange(shape[1]))
ax.set_yticks(np.arange(shape[0]))
ax.set_grid(True, color='black', linestyle='-', linewidth=1)

for r in range(shape[0]):
    for c in range(shape[1]):
        state = r * shape[1] + c
        if r == 3 and 1 <= c <= 10:
            rect = plt.Rectangle((c - 0.5, r - 0.5), 1, 1, facecolor='red', alpha=0.6)
            ax.add_patch(rect)
            ax.text(c, r, "CLIFF", ha='center', va='center', fontweight='bold', color='white', fontsize=9)
        elif r == 3 and c == 0:
            rect = plt.Rectangle((c - 0.5, r - 0.5), 1, 1, facecolor='lightgreen', alpha=0.6)
            ax.add_patch(rect)
            best_action = np.argmax(Q[state])
            ax.text(c, r, f"S\n{action_symbols[best_action]}", ha='center', va='center', fontweight='bold', fontsize=12)
        elif r == 3 and c == 11:
            rect = plt.Rectangle((c - 0.5, r - 0.5), 1, 1, facecolor='gold', alpha=0.7)
            ax.add_patch(rect)
            ax.text(c, r, "GOAL\n🚩", ha='center', va='center', fontweight='bold', fontsize=11)
        else:
            best_action = np.argmax(Q[state])
            ax.text(c, r, action_symbols[best_action], ha='center', va='center', fontsize=16, fontweight='bold', color='darkblue')

ax.set_title("Learned SARSA Policy Grid Map", fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 🎯 8. Evaluate Deterministic Optimal Policy

In [ ]:
eval_env = gym.make("CliffWalking-v1")
state, _ = eval_env.reset()
done = False
total_eval_reward = 0
eval_steps = 0

while not done and eval_steps < 200:
    action = int(np.argmax(Q[state]))
    state, reward, terminated, truncated, _ = eval_env.step(action)
    done = terminated or truncated
    total_eval_reward += reward
    eval_steps += 1

print(f"Deterministic Evaluation Result:")
print(f"  Total Reward : {total_eval_reward}")
print(f"  Total Steps  : {eval_steps}")
eval_env.close()